# Pipeline walkthrough

Loads a sample GPS export from `data/samples/` and runs the real `app` code one step at a time, printing each step's output. The order is the same as `app/etl/pipeline.py`'s `run_pipeline`, followed by what `app/services/ingest_service.py` does with the result. The last cell shows what `GET /exposure/hourly/{pid}` would return for the chart.

To try different thresholds, edit `app/config/pipeline.py` or `app/config/io_types.py`, restart the kernel and run all cells.

`PRAISE_API_KEY`/`PRAISE_MYID` are read from the repo-root `.env` (see `app/config/settings.py`). Only the expo_calx step needs them.

In [ ]:
import os
import sys
from pathlib import Path
from pprint import pprint

# Separate DB file so this notebook never touches the real exposure.db.
# Must be set before anything reads app settings.
os.environ["EXPOSURE_DB_PATH"] = "test_exposure.db"

# This notebook lives in notebooks/, one level below the repo root.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.clients import praise_client
from app.db import exposure_repository
from app.db.connection import init_db
from app.etl.parsers import parse_pings
from app.etl.request_rows import to_request_rows
from app.etl.routes import simplify_routes
from app.etl.spikes import remove_spike_outliers
from app.etl.stays import group_into_stays
from app.etl.time_split import split_by_hour
from app.io_types.registry import label_stays
from app.services.ingest_service import build_db_rows

## Load sample data

A GeoJSON `FeatureCollection` export. Change `DATA_PATH` to try another day.

In [ ]:
import json

DATA_PATH = REPO_ROOT / "data" / "samples" / "20260901.geojson"
PID = DATA_PATH.stem

features = json.loads(DATA_PATH.read_text(encoding="utf-8"))["features"]
print(f"{len(features)} features loaded from {DATA_PATH.name}")
pprint(features[:3])

## Step 1 - `parse_pings`: raw records to time-sorted pings

In [ ]:
pings = parse_pings(features)
print(f"{len(pings)} pings")
pprint(pings[:3])

## Step 2 - `remove_spike_outliers`: drop GPS spikes

In [ ]:
cleaned = remove_spike_outliers(pings)
print(f"{len(cleaned)} pings remain ({len(pings) - len(cleaned)} spikes dropped)")

## Step 3 - `group_into_stays` + `label_stays`

Groups pings into stays (where the person stayed, and for how long), then gives each stay an io type (Home, Other Indoor, Outdoor, ...) using the rules in `app/io_types/`. Labelling happens before the hour split, so a stay that crosses an hour boundary is judged by its full duration.

In [ ]:
from collections import Counter

stays = label_stays(group_into_stays(cleaned))
print(f"{len(stays)} stays:", Counter(s.io for s in stays))
for s in stays[:5]:
    print(s.io, s.start_utc, "->", s.end_utc, f"{s.duration_minutes:.1f} min", (s.lng, s.lat))

## Step 4 - `simplify_routes`: thin moving routes to about one point per few hundred meters

In [ ]:
simplified = simplify_routes(stays)
print(f"{len(stays)} stays -> {len(simplified)} after route simplification")

## Step 5 - `split_by_hour`: cut stays at HK-local clock hours

In [ ]:
slices = split_by_hour(simplified)
print(f"{len(slices)} hourly slices")
pprint(slices[:5])

## Step 6 - `to_request_rows`: build expo_calx rows

In [ ]:
rows = to_request_rows(PID, slices)
print(f"{len(rows)} rows -> {praise_client.count_calls(len(rows))} expo_calx call(s)")
pprint(rows[:5])

## Step 7 - `calculate_exposure`: call expo_calx

Makes real API calls (needs `.env`). Each call takes about 20-30 s.

In [ ]:
results = praise_client.calculate_exposure(rows)
print(f"{len(results)} results")
pprint(results[:5])

## Step 8 - store in the notebook's own DB

In [ ]:
init_db()
exposure_repository.upsert_hourly(build_db_rows(PID, slices, results))

## Step 9 - read back what `GET /exposure/hourly/{pid}` returns

In [ ]:
pprint(exposure_repository.get_hourly(PID))